# zhjudge：在 Kaggle 上训练（GPU T4 x2）

一次跑完整流程：构建数据 → 校验 → 训练 → 校准 → 评测 → 导出（Laya 格式）→ 兼容性检查，最后把导出的模型、评测报告和日志打包到 `/kaggle/working/outputs`。

> 独立开源项目，与 TypeSafe（Jev）和 Convai（Laya）没有任何隶属、合作或背书关系；提及其名称只为说明模型类别（Jev 式类型化决策）和 Laya 检查点格式。不调用 Jev 的接口，也从不用它的输出训练。训练数据只来自 `configs/datasets.yaml` 里 `enabled: true` 的数据集（见 DATA_LICENSES.md）。
> Independent open-source project, not affiliated with or endorsed by TypeSafe (Jev) or Convai (Laya); their names only identify the model category (Jev-style typed decisions) and the Laya checkpoint format. Jev's API is never called and its outputs are never trained on.

**开始前**（逐步点击说明见仓库 README 的「在 Kaggle 上训练」）
1. 右侧 **Session options**：**Accelerator** 选 **GPU T4 x2**，**Internet** 打开（Kaggle 账号要先验证手机号）。
2. 仓库是公开的，**不需要 GitHub 令牌**。只有用私有仓库（例如你用 GitHub 的 Import repository 建的私有副本；设置里把 `REPO` 改成它）时：**Add-ons → Secrets** 添加名为 `GITHUB_TOKEN` 的 Secret（只读的 GitHub fine-grained token：只授权那个仓库，Contents 只读），并勾选本笔记本。令牌只在内存里使用，不写进任何文件、不打印。
3. 正式训练用右上角 **Save Version → Save & Run All (Commit)**：在后台运行，关掉浏览器也会继续。想边看边跑可以用 **Run All**，但浏览器断开太久交互式会话可能被回收。

**检查点与续训**
- 训练每 `save_every` 次参数更新（`train_base.yaml` 为 500）把可续训检查点写到 `/kaggle/working/zhjudge/runs/<run>/ckpt/`（mmBERT-base 约 4 GB，覆盖写入）。
- 每个 Kaggle 会话最多用 `SESSION_HOURS`（默认 11.5 小时，Kaggle 上限 12 小时；从本会话第一次运行设置单元格开始算，同一会话里重新运行不会重新计时，所以交互式会话打开后请尽快运行）：留出 `POST_TRAIN_HOURS` 的收尾时间，训练到点会保存检查点并暂停，然后照常打包输出。
- 同一会话里重新运行全部单元格：自动从 `/kaggle/working` 里的检查点继续。
- 新会话（上一次 Save & Run All 已暂停、失败或完成）：右侧 **Input → Add Input** 挂上本笔记本上一个版本的输出，再运行一次；笔记本会在 `/kaggle/input` 里自动找到用同样配置训练、进度最靠前的检查点（连同当时构建的数据）复制回来继续。如果找到的都是别的配置训练的，笔记本会停下来说明，不会悄悄从头训练。
- 注意：在编辑器里 **Run All** 时，交互式会话一旦因断线被回收，`/kaggle/working` 里的检查点也就没了（除非事先在 Session options 里打开了保留文件的 Persistence），只能从上一个已保存版本的输出继续。所以长时间训练请用 Save & Run All。

**English**: pick *GPU T4 x2* and *Internet on* (the repo is public, no GitHub token needed; only for a private copy set `REPO` to it and add the Secret `GITHUB_TOKEN`, a read-only fine-grained token, and attach it), then *Save Version → Save & Run All (Commit)*. Resumable checkpoints are written to `/kaggle/working/zhjudge/runs`; to continue in a new session attach the previous version's output as an input and run again (the most advanced checkpoint trained with the same config is found automatically; if only checkpoints of a different config are found, the notebook stops and says so). Results: `/kaggle/working/outputs`.

In [ ]:
# 1) 设置：一般只需要改这里 / settings
import os, time
# 本会话第一次运行这个单元格的时间：训练时间预算从这里算，同一会话里重新运行不会重新计时（/tmp 每个新会话都是空的）。
# 交互式会话打开后闲置的时间不算在内，所以打开后请尽快运行。
# First run of this cell in this session: the time budget counts from here, also across re-runs (/tmp is empty in every
# new session). Idle time in an interactive session before the first run is not counted.
try:
    T0 = float(open("/tmp/zhjudge_session_t0").read())
except (OSError, ValueError):
    T0 = time.time()
    open("/tmp/zhjudge_session_t0", "w").write(repr(T0))

REPO = "TerryG907/zhjudge"           # GitHub 仓库（公开；私有副本需要 Secret GITHUB_TOKEN）/ repo
BRANCH = "main"                      # 要训练的分支 / branch to train
CONFIG = "configs/train_base.yaml"   # configs/train_smoke.yaml = 几分钟的流程自检（mmBERT-small，CPU，20 步）
                                     # configs/train_v1b.yaml = v1b（见 README）：作为新版本运行，不要挂 v1 的输出
RUN_NAME = ""                        # 留空 = 配置里的 run.name；续训靠它找检查点，前后几次运行要一致
PRECISION = "fp16"                   # T4 不支持 bf16：fp16 自动混合精度 + GradScaler；连续出现 NaN 时改成 "fp32"
USE_TWO_GPUS = False                 # True = torchrun 在两张 T4 上做数据并行（先做 NCCL 自检，不通过就自动用单卡）
LICENSE_PROFILE = "default"          # default（权重 CC BY-SA 4.0）| permissive_strict（去掉相同方式共享来源，权重 Apache-2.0；
                                     # 改了要从头训练：那一次把 RESUME 设为 False）
SAVE_EVERY = 0                       # 每多少次参数更新存一次可续训检查点；0 = 用配置里的值（train_base.yaml：500）
SESSION_HOURS = 11.5                 # 这个 Kaggle 会话最多用多久（从第一次运行本单元格算；Kaggle 上限 12 小时）
POST_TRAIN_HOURS = 1.0               # 给 校准 + 评测 + 导出 + 打包 预留的时间；训练在此之前暂停
RESUME = True                        # True = 自动从最近的检查点续训；False = 删掉 /kaggle/working 里这个 run 的旧进度、
                                     # 不看 Input，从头训练（只用于一次运行，之后记得改回 True）
EVAL_BASELINE = True                 # 评测时在同一批题目上重测 laya-multilingual（多下载约 650 MB）
COMPARE_PREDS = ""                   # 上一版（如 v0）的 eval_preds.jsonl：评测报告里逐题对比（配对差值 + 95% 区间）。把该文件
                                     # 做成 Kaggle Dataset 挂上，填 /kaggle/input/<数据集>/eval_preds.jsonl；空 = 不对比
COMPARE_LABEL = "v0"                 # 报告里对它的称呼 / its name in the report
MAKE_TARBALL = True                  # 另外把 outputs 打成一个 .tar.gz，方便一次下载
EXTRA_SETS = []                      # 额外的配置覆盖，如 ["train.max_tokens=4096"]（显存不足时）；改 data/model/train
                                     # 下影响训练内容的键会让旧检查点无法续训（train.max_tokens 也算）

WORK = "/kaggle/working"
STATE = f"{WORK}/zhjudge"            # 构建好的数据、训练状态（检查点）和日志：随版本输出保存，用于续训
OUT = f"{WORK}/outputs"              # 最终打包：导出的模型、评测报告、日志
REPO_DIR = "/tmp/zhjudge"            # 代码：每次重新克隆，不进输出
VENV = "/tmp/zhjudge-venv"           # Python 环境（约 6 GB）：放在 /kaggle/working 之外
if not os.path.isdir(WORK):
    raise RuntimeError("这个笔记本只用于 Kaggle（找不到 /kaggle/working）；Colab 请用 notebooks/colab_train.ipynb")

In [ ]:
# 2) 工具函数 + 检查 GPU / helpers + GPU check
import glob, json, shutil, subprocess, sys
from pathlib import Path


def sh(cmd, log=None, cwd=None, env=None):
    """运行命令：输出实时显示在这里，给了 log 就同时追加到该文件。返回退出码。"""
    f = open(log, "a", encoding="utf-8", buffering=1) if log else None
    if f:
        f.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} $ {' '.join(map(str, cmd))}\n")
    p = subprocess.Popen(cmd, cwd=cwd or REPO_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, errors="replace", bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
        if f:
            f.write(line)
    rc = p.wait()
    if f:
        f.write(f"===== exit {rc}\n")
        f.close()
    return rc


smi = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True) if shutil.which("nvidia-smi") else None
GPUS = [l.split(" (UUID")[0] for l in (smi.stdout.splitlines() if smi and smi.returncode == 0 else [])
        if l.startswith("GPU ")]
print("\n".join(GPUS) or "没有看到 NVIDIA GPU / no NVIDIA GPU visible")
if not GPUS and "smoke" not in CONFIG:
    raise RuntimeError("没有 GPU：右侧 Session options → Accelerator 选 GPU T4 x2，然后重新运行")
if GPUS and not all("T4" in g for g in GPUS):
    print("注意：这不是 T4。PRECISION 仍按上面的设置（fp16 + GradScaler），显存不足时调小 train.max_tokens")

In [ ]:
# 3) 克隆代码 / clone the code
#    公开仓库不需要令牌。私有仓库（如私有副本）用 Kaggle Secret GITHUB_TOKEN：令牌只在内存里，通过环境变量交给这一条
#    git 命令，不出现在命令行、不写进 .git/config 或任何文件，也不打印。
#    A public repo needs no token. For a private one the Kaggle Secret GITHUB_TOKEN is used: it stays in memory, is handed
#    to this one git call via the environment, never on a command line, never written to .git/config or any file, never
#    printed.
import base64
from kaggle_secrets import UserSecretsClient


def github_token():
    try:
        return (UserSecretsClient().get_secret("GITHUB_TOKEN") or "").strip()
    except Exception:  # noqa: BLE001  (never print the exception: keep the secret machinery out of the logs)
        return ""


_tok = github_token()
_hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode() if _tok else ""
_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if _tok:
    _env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
shutil.rmtree(REPO_DIR, ignore_errors=True)
_r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPO}.git", REPO_DIR],
                    env=_env, capture_output=True, text=True)
_msg = (_r.stdout + _r.stderr).strip()
_leak = False
if _tok:
    _msg = _msg.replace(_tok, "***").replace(_hdr, "***")
    _leak = os.path.exists(f"{REPO_DIR}/.git/config") and any(
        s in open(f"{REPO_DIR}/.git/config", encoding="utf-8").read() for s in (_tok, _hdr))
_auth = "Secret GITHUB_TOKEN" if _tok else "匿名 / anonymous"
del _tok, _hdr, _env
print(_msg)
if _leak:
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    raise RuntimeError("令牌意外出现在 .git/config 里，已删除克隆")
if _r.returncode != 0:
    raise RuntimeError("克隆失败：检查 Internet 是否打开、REPO 和 BRANCH 是否正确；私有仓库还要检查 Secret GITHUB_TOKEN "
                       "是否存在并勾选了本笔记本、是否过期、是否授权了这个仓库（Contents: Read-only）；公开仓库不需要令牌，"
                       "勾选了过期的 GITHUB_TOKEN 也会导致克隆失败，取消勾选即可")
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print(f"已克隆 / cloned {REPO}@{BRANCH} ({COMMIT}) -> {REPO_DIR}（{_auth}）")

In [ ]:
# 4) 安装依赖：uv 按 uv.lock 装固定版本；scripts/run_all.sh 按 NVIDIA 驱动版本自动选 PyTorch 的 CUDA 版本
#    Install: uv installs the locked versions; run_all.sh picks the torch CUDA build from the driver version
os.environ["UV_PROJECT_ENVIRONMENT"] = VENV
# Kaggle 的 PYTHONPATH 带着它自己的 sitecustomize（依赖 wrapt 等），我们的独立环境里没有这些包：子进程不继承它
# Kaggle's PYTHONPATH carries its own sitecustomize (needs wrapt etc.), which our separate venv lacks: don't inherit it
os.environ.pop("PYTHONPATH", None)
if sh([sys.executable, "-m", "pip", "install", "-q", "uv"], cwd=WORK):
    raise RuntimeError("pip install uv 失败（Internet 是否打开？）")
UV = shutil.which("uv") or os.path.expanduser("~/.local/bin/uv")
if "ZHJUDGE_SYNC_ONLY" not in Path(REPO_DIR, "scripts/run_all.sh").read_text(encoding="utf-8"):
    raise RuntimeError(f"分支 {BRANCH} 的代码比这个笔记本旧（没有 ZHJUDGE_SYNC_ONLY）：请用包含 Kaggle 笔记本的分支")
if sh(["bash", "scripts/run_all.sh"], env=dict(os.environ, ZHJUDGE_SYNC_ONLY="1", UV=UV)):
    raise RuntimeError("依赖安装失败，见上面的输出")

In [ ]:
# 5) 读取配置，找最近的检查点（续训）和已构建的数据 / resolve the config, find the newest checkpoint and data
os.makedirs(f"{STATE}/logs", exist_ok=True)
LOGS = f"{STATE}/logs"
os.environ.update(ZHJUDGE_DATA_DIR=f"{STATE}/data", ZHJUDGE_RUNS_DIR=f"{STATE}/runs", ZHJUDGE_PRECISION=PRECISION,
                  CONFIG=CONFIG, TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1",
                  HF_HUB_DISABLE_PROGRESS_BARS="1", TQDM_DISABLE="1")
if RUN_NAME:
    os.environ["RUN_NAME"] = RUN_NAME
else:
    os.environ.pop("RUN_NAME", None)  # 同一内核里之前设过的不能留着 / drop a value left by an earlier run
SETS = []
if LICENSE_PROFILE != "default":
    SETS += [f"data.license_profile={LICENSE_PROFILE}", "export.license=apache-2.0"]
if SAVE_EVERY:
    SETS.append(f"train.save_every={int(SAVE_EVERY)}")
if not EVAL_BASELINE:
    SETS.append("eval.baseline=false")
SETS += list(EXTRA_SETS)
ARGS = ["--config", CONFIG] + [a for s in SETS for a in ("--set", s)]
ZHJUDGE = [UV, "run", "--no-sync", "zhjudge"]
EVAL_SETS = []  # 只给 eval 步骤 / eval step only (eval.* keys never block a resume)
if COMPARE_PREDS and os.path.isfile(COMPARE_PREDS):
    EVAL_SETS = ["--set", f"eval.compare_preds={COMPARE_PREDS}", "--set", f"eval.compare_label={COMPARE_LABEL}"]
elif COMPARE_PREDS:
    print(f"找不到 COMPARE_PREDS={COMPARE_PREDS}：评测报告里不做逐题对比 / not found: no item-by-item comparison")

# 用和流水线相同的代码解析配置（run 名、要构建的来源等）/ resolve the config with the pipeline's own code
_RESOLVE = r"""
import json, sys
from zhjudge.config import load_config, paths
from zhjudge.data.allowlist import load_allowlist, load_licenses, select_sources
from zhjudge.data.build import SOURCES
from zhjudge.train import resume_hash
from zhjudge.data import build as B
a = sys.argv[1:]
cfg = load_config(a[a.index("--config") + 1], [a[i + 1] for i, x in enumerate(a) if x == "--set"])
P = paths(cfg)
srcs, _ = select_sources(cfg, load_allowlist(P["allowlist"]), SOURCES, load_licenses(P["licenses"]))
print(json.dumps({"run": cfg["run"]["name"], "sources": srcs, "license_profile": cfg["data"].get("license_profile"),
                  "decontaminate": cfg["data"]["decontaminate"], "save_every": cfg["train"]["save_every"],
                  "probes": bool(cfg["data"]["probes"] or cfg["eval"]["probes"]), "base": cfg["model"]["base"],
                  "epochs": cfg["train"]["epochs"], "config_hash": resume_hash(cfg),
                  "corpus_version": getattr(B, "CORPUS_VERSION", 1),
                  "train_sample": cfg["data"].get("train_sample") or {}}))
"""
_r = subprocess.run([UV, "run", "--no-sync", "python", "-c", _RESOLVE, *ARGS], cwd=REPO_DIR, capture_output=True,
                    text=True)
if _r.returncode:
    print(_r.stdout, _r.stderr)
    raise RuntimeError("读取配置失败")
CFG = json.loads(_r.stdout.strip().splitlines()[-1])
RUN = CFG["run"]
RUN_DIR = Path(STATE) / "runs" / RUN
print(f"run {RUN} | base {CFG['base']} | {CFG['epochs']} epoch(s) | {len(CFG['sources'])} sources | "
      f"checkpoint every {CFG['save_every']} updates")


def _read_json(p):
    try:
        return json.loads(Path(p).read_text())
    except Exception:  # noqa: BLE001
        return None


def progress_of(run_dir):
    """(进度, 说明)：2 = 用当前配置训练完了，0–1 = 当前配置的检查点在批次计划里的位置，-1 = 别的配置留下的（不用）；
    什么都没有时返回 None。只认和当前配置同一个 config_hash 的状态（与 train.py 续训时的检查一致）。"""
    d = Path(run_dir)
    done = _read_json(d / "train_done.json")
    if done and (d / "model" / "model.safetensors").exists():
        if done.get("config_hash") == CFG["config_hash"]:
            return 2.0, "训练已完成 / training finished"
    if (d / "ckpt" / "last.pt").exists():
        j = _read_json(d / "ckpt" / "last.json")
        if j is None:
            return 0.0, "检查点（没有 last.json，进度未知；续训时由 train.py 核对配置）"
        if j.get("config_hash") != CFG["config_hash"]:
            return -1.0, "检查点来自不同的配置，不用 / checkpoint from a different config, ignored"
        return (j["batches_done"] / max(1, j["total_batches"]),
                f"检查点 / checkpoint: 更新 {j['updates']}/{j['total_updates']}（{j['world_size']} 张卡），"
                f"已训练 {j['train_elapsed_h']} 小时，保存于 {j['saved_at']}")
    if done:
        return -1.0, "训练结果来自不同的配置，不用 / finished run of a different config, ignored"
    return None


if not RESUME and RUN_DIR.exists():
    shutil.rmtree(RUN_DIR)
    print(f"RESUME = False：已删除 {RUN_DIR}，从头训练")
cands = []
if RESUME:
    if progress_of(RUN_DIR):
        cands.append((*progress_of(RUN_DIR), RUN_DIR))
    for root, dirs, _files in os.walk("/kaggle/input"):
        if root.count(os.sep) >= 10:
            dirs[:] = []
        elif os.path.basename(root) == RUN and os.path.basename(os.path.dirname(root)) == "runs":
            dirs[:] = []
            if progress_of(root):
                cands.append((*progress_of(root), Path(root)))
for p, desc, where in cands:
    print(f"  找到 / found {where}: {desc}")
rejected = [c for c in cands if c[0] < 0]
cands = [c for c in cands if c[0] >= 0]
if cands:
    best = max(cands, key=lambda c: (c[0], c[2] == RUN_DIR))  # 进度相同时优先用本地的 / ties: keep the local one
    if best[2] != RUN_DIR:
        src_state = best[2].parent.parent  # .../zhjudge
        print(f"从 {best[2]} 恢复训练状态（大文件复制需要一两分钟）...")
        shutil.rmtree(RUN_DIR, ignore_errors=True)
        shutil.copytree(best[2], RUN_DIR, ignore=shutil.ignore_patterns("*.tmp"))
        if (src_state / "data" / "unified" / "manifest.json").exists():
            # 同时恢复当时构建的数据，保证批次计划完全一致 / restore the corpus the checkpoint was trained on
            shutil.rmtree(f"{STATE}/data", ignore_errors=True)
            shutil.copytree(src_state / "data", f"{STATE}/data", ignore=shutil.ignore_patterns("*.tmp"))
        for f in glob.glob(f"{src_state}/logs/*"):
            if os.path.isfile(f) and not os.path.exists(f"{LOGS}/{os.path.basename(f)}"):
                shutil.copy2(f, LOGS)
    print("续训起点 / resume from:", best[2], "-", best[1])
elif not RESUME:
    print("RESUME = False：不找 /kaggle/input 里的检查点，从头训练；以后续训前把 RESUME 改回 True")
elif rejected:
    # RESUME = True 时绝不悄悄丢掉进度 / with RESUME = True, progress is never discarded silently
    raise RuntimeError("找到的检查点或训练结果都是用别的配置训练的（上面标了「不用」）。有意改了配置、要从头训练：把 RESUME "
                       "设为 False 再运行；想接着原来的训练：把 CONFIG / LICENSE_PROFILE / EXTRA_SETS（以及分支上的 configs/）"
                       "改回原来的样子")
else:
    print("没有找到检查点：从头训练 / no checkpoint found: training starts from scratch")

# 已有的数据如果和当前配置一致就直接用，否则重新构建 / reuse the built corpus only if it matches this config
# （含 data.train_sample：v1b 的语料和 v1 的不同 / incl. data.train_sample: v1b's corpus is not v1's）
_man = Path(STATE) / "data" / "unified" / "manifest.json"
NEED_BUILD = True
if _man.exists():
    m = json.loads(_man.read_text(encoding="utf-8"))
    NEED_BUILD = not (m.get("corpus_version", 1) == CFG["corpus_version"]
                      and sorted(m.get("sources", {})) == sorted(CFG["sources"])
                      and (m.get("allowlist") or {}).get("license_profile") == CFG["license_profile"]
                      and (m.get("decontamination") or {}).get("mode") == CFG["decontaminate"]
                      and (m.get("train_sample") or {}) == CFG["train_sample"])
NEED_PROBES = CFG["probes"] and (NEED_BUILD or not (Path(STATE) / "data" / "probes" / "zh_probe.jsonl").exists())
print("数据 / data:", "需要构建 / will be built" if NEED_BUILD else f"沿用已构建的 / reusing {_man.parent}")

In [ ]:
# 6) （可选）双卡自检：两张 T4 之间用 NCCL 做 all-reduce，不通过就用单卡 / optional 2-GPU (NCCL) self-check
import signal

WORLD = 1
_CHECK = r"""
import datetime, os, sys, time, torch, torch.distributed as dist
cuda = torch.cuda.is_available()
local = int(os.environ["LOCAL_RANK"])
if cuda:
    torch.cuda.set_device(local)
dist.init_process_group("nccl" if cuda else "gloo", timeout=datetime.timedelta(minutes=2))
dev = torch.device("cuda", local) if cuda else torch.device("cpu")
n = dist.get_world_size()
x = torch.full((64 * 2**20,), float(dist.get_rank() + 1), device=dev)  # 256 MB of fp32
t = time.time()
for _ in range(5):
    y = x.clone()
    dist.all_reduce(y)
if cuda:
    torch.cuda.synchronize()
gbs = 5 * x.numel() * 4 / (time.time() - t) / 1e9
ok = torch.tensor([1.0 if bool((y == float(n * (n + 1) // 2)).all()) else 0.0], device=dev)
dist.all_reduce(ok, op=dist.ReduceOp.MIN)  # every rank got the right sum
ok = bool(ok.item() == 1.0)
dist.barrier(device_ids=[local] if cuda else None)
dist.destroy_process_group()
if os.environ["RANK"] == "0":  # one line only: the ranks' prints would interleave
    print("DDP-CHECK", "OK" if ok else "BAD", f"{n} ranks, all-reduce {gbs:.1f} GB/s", flush=True)
sys.exit(0 if ok else 1)
"""
if USE_TWO_GPUS:
    if len(GPUS) < 2:
        print(f"只看到 {len(GPUS)} 张 GPU（Accelerator 要选 GPU T4 x2）：用单卡训练")
    else:
        # 两张 T4 之间没有 NVLink：关掉 P2P，走主机共享内存，更稳，速度差别不大 / no NVLink between the T4s
        os.environ.setdefault("NCCL_P2P_DISABLE", "1")
        Path("/tmp/zhjudge_ddp_check.py").write_text(_CHECK)
        p = subprocess.Popen([UV, "run", "--no-sync", "torchrun", "--standalone", "--nproc_per_node=2",
                              "/tmp/zhjudge_ddp_check.py"], cwd=REPO_DIR, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, start_new_session=True)
        try:
            out, _ = p.communicate(timeout=300)
        except subprocess.TimeoutExpired:
            # torchrun 的 worker 各在自己的会话里：先 SIGTERM 让 torchrun 收掉它们，不行再逐个强杀
            # torchrun's workers run in their own sessions: SIGTERM lets torchrun stop them; else kill them by name
            os.killpg(p.pid, signal.SIGTERM)
            try:
                out, _ = p.communicate(timeout=60)
            except subprocess.TimeoutExpired:
                os.killpg(p.pid, signal.SIGKILL)
                subprocess.run(["pkill", "-9", "-f", "/tmp/zhjudge_ddp_check.py"])
                try:
                    out, _ = p.communicate(timeout=30)
                except subprocess.TimeoutExpired:
                    out = ""
            out += "\n(timeout: NCCL self-check hung for 5 minutes)"
        print("\n".join(l for l in out.splitlines() if l.startswith(("DDP-CHECK", "(timeout"))) or out[-2000:])
        if p.returncode == 0 and "DDP-CHECK OK" in out:
            WORLD = 2
        else:
            print("双卡自检没通过：用单卡训练 / 2-GPU self-check failed: training on one GPU")
print(f"训练用 {WORLD} 张卡 / training on {WORLD} GPU(s)"
      + ("（每次参数更新看两批数据，更新次数减半，学习率不变）" if WORLD > 1 else ""))

In [ ]:
# 7) 跑流水线 / run the pipeline: (probes) → build → validate → train → calibrate → eval → export → compat
#    训练在 SESSION_HOURS - POST_TRAIN_HOURS 前保存检查点并暂停；任何一步失败都不会中断笔记本，下面照常打包输出。
#    Training saves and pauses before SESSION_HOURS - POST_TRAIN_HOURS; a failed step never stops the notebook, so the
#    outputs (and the checkpoint for resuming) are always saved.
STEPS = []


def step(name, cmd):
    t = time.time()
    print(f"\n==== {name}  {time.strftime('%H:%M:%S')}", flush=True)
    rc = sh(cmd, log=f"{LOGS}/{name}.log")
    STEPS.append({"step": name, "exit": rc, "minutes": round((time.time() - t) / 60, 1)})
    print(f"==== {name}: 退出码 / exit {rc}, {STEPS[-1]['minutes']} min", flush=True)
    return rc


def pipeline():
    if NEED_PROBES and step("probes", ZHJUDGE + ["probes"] + ARGS):
        return "failed:probes"
    if NEED_BUILD and step("build", ZHJUDGE + ["build"] + ARGS):
        return "failed:build"
    if step("validate", ZHJUDGE + ["validate"] + ARGS):
        return "failed:validate"
    hours = SESSION_HOURS - (time.time() - T0) / 3600 - POST_TRAIN_HOURS
    if hours < 0.1:
        print("本次会话剩下的时间不够训练：下次运行再继续 / not enough time left in this session to train")
        return "paused"
    trainer = ([UV, "run", "--no-sync", "torchrun", "--standalone", f"--nproc_per_node={WORLD}", "-m", "zhjudge"]
               if WORLD > 1 else ZHJUDGE)
    t_train = time.time()
    print(f"训练时间预算 / training budget: {hours:.2f} h")
    if step("train", trainer + ["train"] + ARGS + ["--set", f"train.max_hours={hours:.3f}"]):
        last = RUN_DIR / "ckpt" / "last.json"
        info = json.loads(last.read_text()) if last.exists() else {}
        if info.get("reason") == "time_budget" and info.get("saved_unix", 0) >= t_train:
            print("训练到了时间预算：检查点已保存。上面 torchrun 的 ChildFailedError（exitcode 75）是正常的，"
                  "75 就表示「时间到、已保存、可续训」/ time budget reached, checkpoint saved (exit code 75)")
            return "paused"  # 单卡退出码 75；torchrun 会把它报成 1 / torchrun reports the workers' 75 as 1
        return "failed:train"
    for name in ("calibrate", "eval", "export", "compat"):
        if step(name, ZHJUDGE + [name] + ARGS + (EVAL_SETS if name == "eval" else [])):
            return f"failed:{name}"
    return "done"


try:
    STATUS = pipeline()
except Exception as e:  # noqa: BLE001
    STATUS = f"failed:{type(e).__name__}"
    print("流水线出错 / pipeline error:", repr(e))
print("\n状态 / status:", STATUS)

In [ ]:
# 8) 打包到 /kaggle/working/outputs：导出的模型、评测报告、日志 / package the export, reports and logs
import tarfile


def cp(src, dst):
    if os.path.isfile(src):
        shutil.copy2(src, dst)


shutil.rmtree(OUT, ignore_errors=True)
for d in ("model", "reports", "logs"):
    os.makedirs(f"{OUT}/{d}")
# 只打包本次会话里成功跑出来的结果：RUN_DIR 里可能还留着换配置之前的旧导出和旧报告
# only what this session produced: RUN_DIR may still hold an export/reports from an earlier config
RAN = {s["step"] for s in STEPS}
OK = {s["step"] for s in STEPS if s["exit"] == 0}
if "export" in OK:
    for d in sorted(glob.glob(f"{RUN_DIR}/export/*")):
        if os.path.isdir(d):
            shutil.copytree(d, f"{OUT}/model/{os.path.basename(d)}")
for f, step_ok in (("eval.md", "eval" in OK), ("eval.json", "eval" in OK), ("eval_preds.jsonl", "eval" in OK),
                   ("eval_preds_baseline.jsonl", "eval" in OK),
                   ("calibration.json", "calibrate" in OK), ("compat.json", "compat" in RAN),
                   ("train_stats.json", "train" in OK), ("train_done.json", "train" in OK),
                   ("config.resolved.yaml", "train" in RAN)):
    if step_ok:
        cp(f"{RUN_DIR}/{f}", f"{OUT}/reports/{f}")
cp(f"{RUN_DIR}/ckpt/last.json", f"{OUT}/reports/checkpoint_progress.json")
if "validate" in OK:
    cp(f"{STATE}/data/unified/validation.json", f"{OUT}/reports/data_validation.json")
    cp(f"{STATE}/data/unified/manifest.json", f"{OUT}/reports/data_manifest.json")
for f in glob.glob(f"{LOGS}/*.log"):
    cp(f, f"{OUT}/logs/")
for f in ("train_log.jsonl", "dev_log.jsonl"):
    cp(f"{RUN_DIR}/{f}", f"{OUT}/logs/{f}")

_last = RUN_DIR / "ckpt" / "last.json"
CKPT = json.loads(_last.read_text()) if _last.exists() else None
NEXT = {
    "done": "训练、校准、评测、导出都已完成。模型在 outputs/model/，评测报告是 outputs/reports/eval.md。",
    "paused": ("训练因时间预算暂停，检查点已保存在 /kaggle/working/zhjudge/runs。" if CKPT else
               "本次会话的时间在训练开始前就用完了（构建数据太久或 SESSION_HOURS 太小），数据已保存。")
              + "续训：右侧 Input → Add Input 挂上本版本的输出，然后再 Save & Run All 一次（会自动接着做）。",
}.get(STATUS, f"失败于 {STATUS.split(':', 1)[-1]}：原因见 outputs/logs/ 里对应步骤的日志。如果已经存过检查点，修好后"
              "挂上本版本的输出再运行即可继续。")
if STATUS == "paused" and os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Interactive":
    NEXT = ("这是在编辑器里交互式运行的：检查点只在本会话的 /kaggle/working 里，会话结束就会丢失（除非事先打开了"
            "保留文件的 Persistence）。长时间训练请用 Save Version → Save & Run All，暂停后挂上那个版本的输出再运行。")
if not RESUME:
    NEXT += " 注意：RESUME 现在是 False，下次运行前把它改回 True，否则会删掉这次的进度或结果并从头训练。"
status = {"status": STATUS, "next": NEXT, "repo": REPO, "branch": BRANCH, "commit": COMMIT, "config": CONFIG,
          "run": RUN, "sets": SETS, "gpus": GPUS, "world_size": WORLD, "precision": PRECISION, "steps": STEPS,
          "checkpoint": CKPT, "session_hours": round((time.time() - T0) / 3600, 2),
          "finished_at": time.strftime("%Y-%m-%d %H:%M:%S %Z")}
Path(f"{OUT}/status.json").write_text(json.dumps(status, indent=1, ensure_ascii=False), encoding="utf-8")
lines = [f"# zhjudge Kaggle run: {STATUS}", "", NEXT, "",
         f"- 代码 / code: {REPO}@{BRANCH} ({COMMIT}), config `{CONFIG}`, run `{RUN}`" + (f", --set {SETS}" if SETS else ""),
         f"- GPU: {'; '.join(GPUS) or 'none'} | 训练用 {WORLD} 张卡 | precision {PRECISION}",
         f"- 本次会话用时 / session: {status['session_hours']} h", ""]
if CKPT:
    lines += [f"- 检查点进度 / checkpoint: 更新 {CKPT['updates']}/{CKPT['total_updates']}，"
              f"批次 {CKPT['batches_done']}/{CKPT['total_batches']}，保存于 {CKPT['saved_at']}", ""]
lines += ["| 步骤 step | 退出码 exit | 分钟 min |", "|---|---|---|"] + [
    f"| {s['step']} | {s['exit']} | {s['minutes']} |" for s in STEPS]
# 评测集是否与 v0 逐条相同（validate 的警告），以及评测报告附加部分的出错情况 / eval sets vs v0, report additions
_v0 = ((_read_json(f"{STATE}/data/unified/validation.json") or {}) if "validate" in OK else {}).get("eval_set_vs_v0") or {}
if _v0.get("same_as_v0") is not None:
    _d = {k: _v0[k] for k in ("count_differences", "dropped_eval_overlapping_probes") if _v0.get(k)}
    lines += ["", "- 评测集与 v0 / eval sets vs v0: " + (
        "各来源 dev/test 条数相同，没有因与探针重叠去掉的样本 / same per-source counts, no probe-overlap drops"
        if _v0["same_as_v0"] else "不同，这些来源的 test:* 与 v0 不是同一批题目 / differ, test:* of these sources is "
        "not item-identical to v0: " + json.dumps(_d, ensure_ascii=False)[:800])]
_ev = (_read_json(RUN_DIR / "eval.json") or {}) if "eval" in OK else {}
if _ev.get("report_errors"):
    lines += [f"- 评测报告附加部分出错（主报告不受影响）/ report additions that failed: {', '.join(_ev['report_errors'])}"]
Path(f"{OUT}/STATUS.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

# 安全检查：输出里不能有 GitHub 令牌（照理不会有）。发现就删除该文件并报告 / make sure no file contains the token
def token_scan():
    tok = github_token()
    if not tok:
        return None
    needles = [tok.encode(), base64.b64encode(f"x-access-token:{tok}".encode())]
    del tok
    hits = []
    for root, _dirs, files in os.walk(WORK):
        for fn in files:
            p = os.path.join(root, fn)
            if fn.endswith((".pt", ".safetensors", ".pkl", ".tmp")) or os.path.getsize(p) > 256 * 2**20:
                continue  # 权重、检查点、分词缓存 / weights, checkpoints, token caches
            with open(p, "rb") as f:
                data = f.read()
            if any(n in data for n in needles):
                hits.append(p)
    for p in hits:
        os.remove(p)
    return hits


_hits = token_scan()
if _hits:
    print("⚠ 发现并删除了含有令牌的文件（请到 GitHub 作废这个 token 并换一个新的）:", _hits)
else:
    print("令牌检查 / token scan:", "通过 / clean" if _hits == [] else "跳过（没有使用 GITHUB_TOKEN）/ skipped (no token used)")

if MAKE_TARBALL:
    tgz = f"{OUT}/zhjudge-{RUN}-outputs.tar.gz"
    with tarfile.open(tgz, "w:gz", compresslevel=1) as tar:
        for name in ("STATUS.md", "status.json", "model", "reports", "logs"):
            tar.add(f"{OUT}/{name}", arcname=f"zhjudge-{RUN}/{name}")
print(f"已打包到 / packaged into {OUT}")

In [ ]:
# 9) 结果 / results
print(Path(f"{OUT}/STATUS.md").read_text(encoding="utf-8"))
if os.path.exists(f"{OUT}/reports/eval.md"):
    print(Path(f"{OUT}/reports/eval.md").read_text(encoding="utf-8"))
for p in sorted(Path(OUT).rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 2**20:9.1f} MB  {p.relative_to(WORK)}")

### 下一步 / next

- **状态 done**：到本版本的 **Output** 页下载 `outputs/`（或其中的 `zhjudge-<run>-outputs.tar.gz`）：`model/` 是 Laya 格式的检查点（`laya` 参考运行时可直接加载，见 `reports/compat.json`；`laya-mlx` 只在 Apple 芯片上检查），`reports/eval.md` 是评测报告，`logs/` 是各步骤日志。
- **状态 paused**（Save & Run All 的版本）：右侧 **Input → Add Input** 挂上本笔记本刚保存的版本输出（已经挂过的话，确认它指向最新版本），再 **Save Version → Save & Run All**，会自动从检查点继续。交互式 Run All 暂停时检查点只在本会话里，会话结束就没了。
- **状态 failed:…**：看 `outputs/logs/<步骤>.log`。常见问题：显存不足 → `EXTRA_SETS = ["train.max_tokens=4096"]`（要从头训练）；T4 上连续 NaN → `PRECISION = "fp32"`；双卡卡住 → `USE_TWO_GPUS = False`。
- `/kaggle/working/zhjudge/` 里是构建好的数据、训练状态（检查点）和日志，续训要用，不要删。